# 08: Local RAG ด้วย LlamaIndex
    แบบฝึกหัดสำหรับผู้เข้าร่วม Sri Trang

    เลือก Python kernel แล้วกด **Run All** cell แรกจะติดตั้ง packages ให้
    สำหรับการรัน local ให้เปิด Ollama ที่มีโมเดลตาม README ไว้ก่อน
    โค้ด HTTP API ใช้ได้ทั้ง local และ Colab แต่ `localhost` ใน Colab หมายถึงเครื่อง Colab
    หากเรียก Ollama จาก Colab ต้องกำหนด HOST เป็น endpoint ที่ runtime เข้าถึงได้
    โค้ดและข้อมูลทั้งหมดอยู่ใน cells ไม่ต้องเตรียมหรือแก้ไฟล์ประกอบ
    ตัวอย่างทั้งหมดเป็นข้อมูลสมมติสำหรับการเรียน ไม่ใช่ข้อมูลหรือ SOP จริงของบริษัท


## ติดตั้ง packages
ติดตั้งเฉพาะ packages ของ RAG ส่วนโมเดล Ollama เตรียมไว้ตาม README


In [ ]:
%pip install -q "llama-index-core>=0.14,<0.15" "llama-index-embeddings-ollama>=0.10,<0.11" "llama-index-llms-ollama>=0.9,<0.10"


## 1. กำหนดโมเดล local
ใช้ bge-m3 สร้าง embeddings และ qwen3:4b สร้างคำตอบ


In [ ]:
from llama_index.core import Document, VectorStoreIndex, PromptTemplate
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.llms.ollama import Ollama

HOST = "http://127.0.0.1:11434"
embed_model = OllamaEmbedding(model_name="bge-m3", base_url=HOST)
llm = Ollama(model="qwen3:4b", base_url=HOST, request_timeout=180,
             context_window=4096, temperature=0, thinking=False)


## 2. เตรียมเอกสารโรงงานสมมติ
ข้อความนี้เป็นเอกสารต้นทางสำหรับค้นข้อมูล ไม่ใช่ผล retrieval ที่เลือกไว้ล่วงหน้า


In [ ]:
factory_notes = """เอกสารโรงงานยางสมมติสำหรับการเรียน

การรับน้ำยางเข้าคลัง
เมื่อรับน้ำยาง ให้บันทึกรหัสล็อต ชื่อผู้ส่งมอบ น้ำหนักรับเข้า และเวลารับเข้า
เพื่อให้ทีมคลังและทีม QC ตรวจสอบย้อนกลับถึงผู้ส่งมอบได้ หากเอกสารผู้ส่งมอบไม่ครบ
ให้บันทึกว่าข้อมูลขาดและส่งให้ผู้รับผิดชอบติดตาม อย่าเติมข้อมูลที่ยังไม่ได้รับ

การบันทึกข้อมูล QC
บันทึกรหัสตัวอย่าง QC ที่เชื่อมกับรหัสล็อต บันทึกเฉพาะผลที่ทีม QC ยืนยันแล้ว
หากยังไม่มีผลตรวจ ให้ระบุว่ารอผลและไม่กรอกค่าผลตรวจเอง
เอกสารนี้ไม่มีค่า DRC หรือผลตรวจจริงของล็อตใด

รายงานส่งมอบกะ
รายงานระบุปริมาณรับเข้ารวม รหัสล็อตที่รอผล QC และเอกสารผู้ส่งมอบที่ยังไม่ครบ
ระบุเวลาที่อัปเดตรายงาน และแยกข้อมูลที่ยืนยันแล้วออกจากงานที่รอติดตาม

ข้อมูลขนส่ง
บันทึกรหัสเที่ยวรถ วันเวลานัดรับ และรหัสล็อตที่เกี่ยวข้อง
หากรถล่าช้าให้บันทึกเวลาล่าช้าและแจ้งผู้ประสานงาน
เอกสารนี้ไม่มีตารางเดินรถจริงหรือราคาค่าขนส่ง"""

documents = [Document(text=factory_notes, metadata={"source": "เอกสารโรงงานสมมติ"})]


## 3. สร้าง vector index
LlamaIndex แบ่งเอกสารเป็น chunks แล้วสร้าง embeddings ด้วย Ollama เก็บ index ในหน่วยความจำ


In [ ]:
splitter = SentenceSplitter(chunk_size=180, chunk_overlap=30)
nodes = splitter.get_nodes_from_documents(documents)
for i, node in enumerate(nodes, start=1):
    node.metadata["chunk_id"] = f"FACTORY-{i:02d}"
index = VectorStoreIndex(nodes, embed_model=embed_model, show_progress=False)
retriever = index.as_retriever(similarity_top_k=2)


## 4. ค้นหลักฐานที่เกี่ยวข้อง
คำถามจะถูกแปลงเป็น embedding เพื่อค้น chunks ที่ใกล้เคียง อ่านหลักฐานที่ค้นได้ด้านล่าง


In [ ]:
question = "ต้องบันทึกข้อมูลอะไรเมื่อรับน้ำยางเข้าคลัง?"
retrieved = retriever.retrieve(question)
context = "\n\n".join(
    f"[{hit.node.metadata['chunk_id']}] {hit.node.text}" for hit in retrieved
)
print(context)


## 5. สร้างคำตอบจากหลักฐาน
ให้โมเดลตอบเฉพาะข้อมูลใน context พร้อมอ้างอิง chunk ID


In [ ]:
qa_prompt = PromptTemplate("""ตอบภาษาไทยจากหลักฐานของโรงงานสมมติเท่านั้น
อ้างอิง chunk ID เช่น [FACTORY-01] หากข้อมูลไม่พอให้ตอบว่าไม่พบข้อมูลในเอกสาร
ห้ามเดาตัวเลขหรือผลตรวจ

หลักฐาน:
{context_str}

คำถาม: {query_str}
คำตอบ:""")
answer = llm.complete(qa_prompt.format(context_str=context, query_str=question))
print(answer.text)


## เสร็จแล้ว
ลองเปลี่ยนคำถามในขั้นตอน 4 แล้วรันขั้นตอน 4–5 อีกครั้ง เช่น ‘รายงานส่งมอบกะต้องมีข้อมูลอะไร?’ หรือ ‘ล็อต L-101 มีค่า DRC เท่าไร?’

เปลี่ยนเฉพาะโมเดลสร้างคำตอบสามารถใช้ index เดิมได้ หากเปลี่ยนเอกสารหรือ embedding model ให้สร้าง index ใหม่
